# 03 — EDA & Insight Discovery



In [37]:
# ============================================================
# 0. Setup, paths, and data loading
# ============================================================

from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 150)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

# Resolve project root whether this notebook is run from /notebooks or project root
CWD = Path.cwd()
PROJECT_ROOT = CWD.parent if CWD.name == "notebooks" else CWD

RAW_PATH = PROJECT_ROOT / "data" / "raw" / "mobile_usage_behavioral_analysis.csv"
FEATURE_PATH = PROJECT_ROOT / "data" / "processed" / "smartphone_features.csv"
INSIGHT_DIR = PROJECT_ROOT / "reports" / "insights"
FIG_DIR = PROJECT_ROOT / "reports" / "figures" / "eda"
INSIGHT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

AGE_ORDER = ["18–24", "25–34", "35–44", "45–54", "55–59"]
SEGMENT_ORDER = ["Light (<4h)", "Moderate (4–8h)", "Heavy (8–12h)", "Extreme (>12h)"]
LIFESTYLE_ORDER = ["Social-dominant", "Productivity-dominant", "Gaming-dominant"]
USAGE_COLS = [
    "Total_App_Usage_Hours",
    "Daily_Screen_Time_Hours",
    "Number_of_Apps_Used",
    "Social_Media_Usage_Hours",
    "Productivity_App_Usage_Hours",
    "Gaming_App_Usage_Hours",
]
CATEGORY_COLS = [
    "Social_Media_Usage_Hours",
    "Productivity_App_Usage_Hours",
    "Gaming_App_Usage_Hours",
]

def age_group(age):
    if age <= 24:
        return "18–24"
    if age <= 34:
        return "25–34"
    if age <= 44:
        return "35–44"
    if age <= 54:
        return "45–54"
    return "55–59"

def screen_segment(hours):
    if hours < 4:
        return "Light (<4h)"
    if hours < 8:
        return "Moderate (4–8h)"
    if hours < 12:
        return "Heavy (8–12h)"
    return "Extreme (>12h)"

def dominant_lifestyle(row):
    values = {
        "Social-dominant": row["Social_Media_Usage_Hours"],
        "Productivity-dominant": row["Productivity_App_Usage_Hours"],
        "Gaming-dominant": row["Gaming_App_Usage_Hours"],
    }
    return max(values, key=values.get)

def add_minimum_features(df):
    """Fallback only: keep this notebook runnable even if 02 has not been executed yet."""
    df = df.copy()
    if "Age_Group" not in df.columns:
        df["Age_Group"] = df["Age"].apply(age_group)
    if "Screen_Time_Segment" not in df.columns:
        df["Screen_Time_Segment"] = df["Daily_Screen_Time_Hours"].apply(screen_segment)
    if "Dominant_Lifestyle" not in df.columns:
        df["Dominant_Lifestyle"] = df.apply(dominant_lifestyle, axis=1)
    if "Entertainment_Hours" not in df.columns:
        df["Entertainment_Hours"] = df["Social_Media_Usage_Hours"] + df["Gaming_App_Usage_Hours"]
    if "App_Fragmentation_Score" not in df.columns:
        df["App_Fragmentation_Score"] = df["Total_App_Usage_Hours"] / df["Number_of_Apps_Used"].replace(0, np.nan)
    if "Category_Usage_Sum" not in df.columns:
        df["Category_Usage_Sum"] = df[CATEGORY_COLS].sum(axis=1)
    return df

if FEATURE_PATH.exists():
    df = pd.read_csv(FEATURE_PATH)
    data_source = FEATURE_PATH
else:
    df = pd.read_csv(RAW_PATH)
    df = add_minimum_features(df)
    data_source = RAW_PATH

# Ordered categorical types for stable visual ordering
if "Age_Group" in df.columns:
    df["Age_Group"] = pd.Categorical(df["Age_Group"], categories=AGE_ORDER, ordered=True)
if "Screen_Time_Segment" in df.columns:
    df["Screen_Time_Segment"] = pd.Categorical(df["Screen_Time_Segment"], categories=SEGMENT_ORDER, ordered=True)

print(f"Loaded data from: {data_source}")
print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]:,} columns")
display(df.head())

Loaded data from: /Users/hoangquannguyen/Documents/Project/Dataviz/Group14_IT4023E_Dataviz_project/data/processed/smartphone_features.csv
Shape: 1,000 rows × 21 columns


,User_ID,Age,Gender,Total_App_Usage_Hours,Daily_Screen_Time_Hours,Number_of_Apps_Used,Social_Media_Usage_Hours,Productivity_App_Usage_Hours,Gaming_App_Usage_Hours,Location,Age_Group,Screen_Time_Segment,Dominant_Lifestyle,Category_Usage_Sum,Entertainment_Hours,Screen_to_App_Gap,App_Fragmentation_Score,Category_vs_Total_Diff,Social_Ratio,Productivity_Ratio,Gaming_Ratio
0,1,56,Male,2.610,7.150,24,4.430,0.550,2.400,Los Angeles,55–59,Moderate (4–8h),Social Enthusiast,7.380,6.830,4.540,0.109,4.770,0.600,0.075,0.325
1,2,46,Male,2.130,13.790,18,4.670,4.420,2.430,Chicago,45–54,Extreme (>12h),Social Enthusiast,11.520,7.100,11.660,0.118,9.390,0.405,0.384,0.211
2,3,32,Female,7.280,4.500,11,4.580,1.710,2.830,Houston,25–34,Moderate (4–8h),Social Enthusiast,9.120,7.410,-2.780,0.662,1.840,0.502,0.187,0.310
3,4,25,Female,1.200,6.290,21,3.180,3.420,4.580,Phoenix,25–34,Moderate (4–8h),Mobile Gamer,11.180,7.760,5.090,0.057,9.980,0.284,0.306,0.410
4,5,38,Male,6.310,12.590,14,3.150,0.130,4.000,New York,35–44,Extreme (>12h),Mobile Gamer,7.280,7.150,6.280,0.451,0.970,0.433,0.018,0.549


## 1. EDA framing

This notebook does not repeat basic data audit checks. Instead, it asks four exploratory questions:

1. **Usage intensity:** how much do users use smartphones, and how concentrated are heavy users?
2. **Lifestyle pattern:** do users behave differently across social, productivity, and gaming activities?
3. **Demographic / location differences:** where are group differences visible, and where are they weak?
4. **Behavioral relationships:** which variables move together, and which assumptions are not strongly supported?


In [38]:
# ============================================================
# 1. Executive metrics and insight hook
# ============================================================

n_users = len(df)
avg_screen = df["Daily_Screen_Time_Hours"].mean()
avg_app_usage = df["Total_App_Usage_Hours"].mean()
avg_apps = df["Number_of_Apps_Used"].mean()
median_screen = df["Daily_Screen_Time_Hours"].median()
heavy_extreme_mask = df["Screen_Time_Segment"].isin(["Heavy (8–12h)", "Extreme (>12h)"])
heavy_extreme_users = int(heavy_extreme_mask.sum())
heavy_extreme_rate = heavy_extreme_users / n_users * 100

summary_kpi = pd.DataFrame({
    "Metric": [
        "Total users",
        "Average daily screen time",
        "Median daily screen time",
        "Average total app usage",
        "Average number of apps used",
        "Heavy / extreme users",
        "Heavy / extreme user rate",
    ],
    "Value": [
        f"{n_users:,}",
        f"{avg_screen:.2f} hours/day",
        f"{median_screen:.2f} hours/day",
        f"{avg_app_usage:.2f} hours",
        f"{avg_apps:.2f} apps",
        f"{heavy_extreme_users:,} users",
        f"{heavy_extreme_rate:.1f}%",
    ]
})

display(summary_kpi)

segment_summary = (
    df["Screen_Time_Segment"]
    .value_counts()
    .reindex(SEGMENT_ORDER)
    .rename_axis("Screen_Time_Segment")
    .reset_index(name="Users")
)
segment_summary["Share_%"] = (segment_summary["Users"] / n_users * 100).round(1)

display(segment_summary)

fig = px.bar(
    segment_summary,
    x="Users",
    y="Screen_Time_Segment",
    orientation="h",
    text="Share_%",
    title="Screen-time segments: heavy and extreme users form the main behavioral concern",
    labels={"Screen_Time_Segment": "Screen-time segment", "Users": "Number of users"},
    category_orders={"Screen_Time_Segment": SEGMENT_ORDER},
)
fig.update_traces(texttemplate="%{text:.1f}%", textposition="outside")
fig.update_layout(height=390, yaxis={"categoryorder": "array", "categoryarray": SEGMENT_ORDER[::-1]})
fig.show()

print(
    f"Opening insight candidate: {heavy_extreme_rate:.1f}% of users are Heavy or Extreme users. "
    "This is stronger than simply reporting average screen time because it reveals concentration in high-usage groups."
)

,Metric,Value
0,Total users,"1,000"
1,Average daily screen time,7.70 hours/day
2,Median daily screen time,7.88 hours/day
3,Average total app usage,6.41 hours
4,Average number of apps used,16.65 apps
5,Heavy / extreme users,488 users
6,Heavy / extreme user rate,48.8%


,Screen_Time_Segment,Users,Share_%
0,Light (<4h),217,21.700
1,Moderate (4–8h),295,29.500
2,Heavy (8–12h),331,33.100
3,Extreme (>12h),157,15.700


Opening insight candidate: 48.8% of users are Heavy or Extreme users. This is stronger than simply reporting average screen time because it reveals concentration in high-usage groups.


In [39]:
# ============================================================
# 2. Distribution deep dive: center, spread, tails, and segment boundaries
# ============================================================

screen_stats = df["Daily_Screen_Time_Hours"].describe(percentiles=[0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95]).to_frame("Daily_Screen_Time_Hours")
usage_stats = df[USAGE_COLS].describe(percentiles=[0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95]).T.round(3)

display(screen_stats)
display(usage_stats)

fig = go.Figure()
for seg in SEGMENT_ORDER:
    sub = df[df["Screen_Time_Segment"] == seg]
    fig.add_trace(go.Histogram(
        x=sub["Daily_Screen_Time_Hours"],
        name=seg,
        nbinsx=35,
        opacity=0.72,
    ))

fig.add_vline(x=4, line_dash="dash", annotation_text="Moderate threshold")
fig.add_vline(x=8, line_dash="dash", annotation_text="Heavy threshold")
fig.add_vline(x=12, line_dash="dash", annotation_text="Extreme threshold")
fig.update_layout(
    title="Daily screen time distribution with behavioral thresholds",
    xaxis_title="Daily screen time (hours)",
    yaxis_title="User count",
    barmode="overlay",
    height=450,
)
fig.show()

# Distribution by gender as an EDA check, not final design
fig = px.violin(
    df,
    x="Gender",
    y="Daily_Screen_Time_Hours",
    box=True,
    points=False,
    title="Screen-time distribution by gender: compare spread, not just averages",
    labels={"Daily_Screen_Time_Hours": "Daily screen time (hours)"},
)
fig.update_layout(height=420)
fig.show()

# Heavy-user rate by group for fast insight scan
heavy_by_age = (
    df.assign(Is_Heavy_Extreme=heavy_extreme_mask)
    .groupby("Age_Group", observed=False)["Is_Heavy_Extreme"]
    .mean()
    .mul(100)
    .reindex(AGE_ORDER)
    .reset_index(name="Heavy_Extreme_Rate_%")
)
display(heavy_by_age.round(2))

,Daily_Screen_Time_Hours
count,"1,000.000"
mean,7.696
std,3.714
min,1.010
5%,1.580
10%,2.358
25%,4.530
50%,7.880
75%,10.910
90%,12.641


,count,mean,std,min,5%,10%,25%,50%,75%,90%,95%,max
Total_App_Usage_Hours,"1,000.000",6.406,3.135,1.000,1.590,2.149,3.590,6.455,9.122,10.680,11.431,11.970
Daily_Screen_Time_Hours,"1,000.000",7.696,3.714,1.010,1.580,2.358,4.530,7.880,10.910,12.641,13.271,14.000
Number_of_Apps_Used,"1,000.000",16.647,7.620,3.000,4.000,6.000,10.000,17.000,23.000,27.000,28.000,29.000
Social_Media_Usage_Hours,"1,000.000",2.456,1.440,0.000,0.250,0.480,1.200,2.445,3.672,4.450,4.730,4.990
Productivity_App_Usage_Hours,"1,000.000",2.495,1.443,0.000,0.270,0.519,1.282,2.435,3.710,4.550,4.800,5.000
Gaming_App_Usage_Hours,"1,000.000",2.475,1.450,0.010,0.280,0.468,1.220,2.455,3.782,4.530,4.740,5.000


,Age_Group,Heavy_Extreme_Rate_%
0,18–24,54.440
1,25–34,48.000
2,35–44,47.840
3,45–54,46.420
4,55–59,49.540


In [40]:
# ============================================================
# 3. Lifestyle pattern: dominant behavior and category mix
# ============================================================

# Auto-detect actual lifestyle labels from the dataset
actual_lifestyles = df["Dominant_Lifestyle"].dropna().unique().tolist()
print("Actual lifestyle labels:", actual_lifestyles)

# Use preferred order only if labels exist in data
preferred_order = [
    "Social Enthusiast",
    "Productivity Focused",
    "Mobile Gamer",
    "Social-dominant",
    "Productivity-dominant",
    "Gaming-dominant",
]

LIFESTYLE_ORDER_FIXED = [
    x for x in preferred_order if x in actual_lifestyles
] + [
    x for x in actual_lifestyles if x not in preferred_order
]

lifestyle_summary = (
    df["Dominant_Lifestyle"]
    .value_counts()
    .rename_axis("Dominant_Lifestyle")
    .reset_index(name="Users")
)

lifestyle_summary["Dominant_Lifestyle"] = pd.Categorical(
    lifestyle_summary["Dominant_Lifestyle"],
    categories=LIFESTYLE_ORDER_FIXED,
    ordered=True
)

lifestyle_summary = lifestyle_summary.sort_values("Dominant_Lifestyle")
lifestyle_summary["Share_%"] = (
    lifestyle_summary["Users"] / n_users * 100
).round(1)

display(lifestyle_summary)

fig = px.bar(
    lifestyle_summary.sort_values("Users"),
    x="Users",
    y="Dominant_Lifestyle",
    orientation="h",
    text="Share_%",
    title="Dominant digital lifestyle groups are relatively balanced",
    labels={"Dominant_Lifestyle": "Dominant lifestyle", "Users": "Number of users"},
)
fig.update_traces(texttemplate="%{text:.1f}%", textposition="outside")
fig.update_layout(height=390)
fig.show()


# Important EDA note:
# Category columns are best treated as behavioral intensities,
# not exact parts of Total_App_Usage.
category_sum = df[CATEGORY_COLS].sum(axis=1)
category_consistency = pd.DataFrame({
    "Metric": [
        "Average Total_App_Usage_Hours",
        "Average Social + Productivity + Gaming",
        "Rows where category sum > total app usage",
        "Share of rows where category sum > total app usage",
    ],
    "Value": [
        round(df["Total_App_Usage_Hours"].mean(), 2),
        round(category_sum.mean(), 2),
        int((category_sum > df["Total_App_Usage_Hours"]).sum()),
        f"{(category_sum > df['Total_App_Usage_Hours']).mean() * 100:.1f}%",
    ]
})
display(category_consistency)


# Normalize only within the 3 category columns to study relative behavior mix.
mix = df.copy()
mix["Category_Usage_Sum"] = mix[CATEGORY_COLS].sum(axis=1)

for col in CATEGORY_COLS:
    share_col = col.replace("_Usage_Hours", "_Share")
    mix[share_col] = mix[col] / mix["Category_Usage_Sum"].replace(0, np.nan)

mix_summary = (
    mix.groupby("Dominant_Lifestyle", observed=False)[[
        "Social_Media_Share",
        "Productivity_App_Share",
        "Gaming_App_Share",
        "Daily_Screen_Time_Hours",
        "Number_of_Apps_Used",
    ]]
    .mean()
    .round(3)
    .reset_index()
)

mix_summary["Dominant_Lifestyle"] = pd.Categorical(
    mix_summary["Dominant_Lifestyle"],
    categories=LIFESTYLE_ORDER_FIXED,
    ordered=True
)

mix_summary = mix_summary.sort_values("Dominant_Lifestyle")

display(mix_summary)


fig = px.scatter_ternary(
    mix.sample(min(600, len(mix)), random_state=42),
    a="Social_Media_Share",
    b="Productivity_App_Share",
    c="Gaming_App_Share",
    color="Dominant_Lifestyle",
    hover_data=["Age", "Gender", "Location", "Daily_Screen_Time_Hours"],
    title="Relative category mix: each point is a user normalized across social, productivity, and gaming",
)
fig.update_layout(height=560)
fig.show()

Actual lifestyle labels: ['Social Enthusiast', 'Mobile Gamer', 'Productivity Focused']


,Dominant_Lifestyle,Users,Share_%
1,Social Enthusiast,338,33.800
0,Productivity Focused,349,34.900
2,Mobile Gamer,313,31.300


,Metric,Value
0,Average Total_App_Usage_Hours,6.410
1,Average Social + Productivity + Gaming,7.430
2,Rows where category sum > total app usage,588
3,Share of rows where category sum > total app u...,58.8%


,Dominant_Lifestyle,Social_Media_Share,Productivity_App_Share,Gaming_App_Share,Daily_Screen_Time_Hours,Number_of_Apps_Used
2,Social Enthusiast,0.530,0.224,0.246,7.678,16.932
1,Productivity Focused,0.227,0.535,0.238,7.898,16.430
0,Mobile Gamer,0.234,0.244,0.522,7.491,16.581


In [41]:
# ============================================================
# 4. Demographic and location differences: where differences are visible
# ============================================================

def group_summary(group_col):
    return (
        df.groupby(group_col, observed=False)
        .agg(
            Users=("User_ID", "count"),
            Avg_Screen_Time=("Daily_Screen_Time_Hours", "mean"),
            Median_Screen_Time=("Daily_Screen_Time_Hours", "median"),
            Avg_Total_App_Usage=("Total_App_Usage_Hours", "mean"),
            Avg_Apps_Used=("Number_of_Apps_Used", "mean"),
            Avg_Social=("Social_Media_Usage_Hours", "mean"),
            Avg_Productivity=("Productivity_App_Usage_Hours", "mean"),
            Avg_Gaming=("Gaming_App_Usage_Hours", "mean"),
        )
        .reset_index()
        .round(2)
    )

age_summary = group_summary("Age_Group")
location_summary = group_summary("Location").sort_values("Avg_Screen_Time", ascending=False)
gender_summary = group_summary("Gender")

display(age_summary)
display(location_summary)
display(gender_summary)

# Age × category heatmap
age_heatmap = age_summary.set_index("Age_Group")[["Avg_Social", "Avg_Productivity", "Avg_Gaming", "Avg_Total_App_Usage", "Avg_Screen_Time"]]
fig = px.imshow(
    age_heatmap,
    text_auto=True,
    aspect="auto",
    title="Average usage hours by age group: heatmap reveals category-level differences",
    labels={"color": "Average hours"},
)
fig.update_layout(height=430)
fig.show()

# Location ranking by screen time: length on common scale is more accurate than area/pie.
fig = px.bar(
    location_summary.sort_values("Avg_Screen_Time"),
    x="Avg_Screen_Time",
    y="Location",
    orientation="h",
    text="Avg_Screen_Time",
    title="Average daily screen time by location",
    labels={"Avg_Screen_Time": "Average daily screen time (hours)"},
)
fig.update_traces(texttemplate="%{text:.2f}h", textposition="outside")
fig.update_layout(height=400)
fig.show()

# Dominant lifestyle by age group: normalized stacked bars
lifestyle_age = (
    df.groupby(["Age_Group", "Dominant_Lifestyle"], observed=False)
    .size()
    .reset_index(name="Users")
)
lifestyle_age["Share_%"] = lifestyle_age.groupby("Age_Group", observed=False)["Users"].transform(lambda s: s / s.sum() * 100)

fig = px.bar(
    lifestyle_age,
    x="Age_Group",
    y="Share_%",
    color="Dominant_Lifestyle",
    category_orders={"Age_Group": AGE_ORDER, "Dominant_Lifestyle": LIFESTYLE_ORDER},
    title="Lifestyle composition by age group",
    labels={"Share_%": "Share of age group (%)"},
)
fig.update_layout(height=430, yaxis_ticksuffix="%")
fig.show()

# Group difference scan: max-min effect sizes for prioritizing dashboard comparisons
comparison_scan = []
for group_col in ["Age_Group", "Gender", "Location"]:
    tmp = group_summary(group_col)
    for metric in ["Avg_Screen_Time", "Avg_Total_App_Usage", "Avg_Apps_Used", "Avg_Social", "Avg_Productivity", "Avg_Gaming"]:
        comparison_scan.append({
            "Group": group_col,
            "Metric": metric,
            "Min": tmp[metric].min(),
            "Max": tmp[metric].max(),
            "Range": tmp[metric].max() - tmp[metric].min(),
            "Highest_Group": tmp.loc[tmp[metric].idxmax(), group_col],
            "Lowest_Group": tmp.loc[tmp[metric].idxmin(), group_col],
        })
comparison_scan = pd.DataFrame(comparison_scan).sort_values("Range", ascending=False)
display(comparison_scan.head(15).round(3))

,Age_Group,Users,Avg_Screen_Time,Median_Screen_Time,Avg_Total_App_Usage,Avg_Apps_Used,Avg_Social,Avg_Productivity,Avg_Gaming
0,18–24,169,8.050,8.580,6.560,16.260,2.560,2.460,2.430
1,25–34,225,7.570,7.710,6.250,17.240,2.480,2.370,2.490
2,35–44,232,7.650,7.620,6.400,16.170,2.250,2.690,2.610
3,45–54,265,7.640,7.840,6.440,16.540,2.490,2.510,2.310
4,55–59,109,7.670,7.910,6.420,17.300,2.590,2.360,2.660


,Location,Users,Avg_Screen_Time,Median_Screen_Time,Avg_Total_App_Usage,Avg_Apps_Used,Avg_Social,Avg_Productivity,Avg_Gaming
0,Chicago,192,8.140,8.430,5.980,17.200,2.390,2.420,2.500
4,Phoenix,199,7.950,8.050,6.570,16.360,2.490,2.580,2.470
2,Los Angeles,185,7.600,7.900,6.460,17.400,2.490,2.580,2.460
1,Houston,181,7.450,7.760,6.340,15.930,2.550,2.500,2.460
3,New York,243,7.400,7.220,6.610,16.410,2.390,2.420,2.480


,Gender,Users,Avg_Screen_Time,Median_Screen_Time,Avg_Total_App_Usage,Avg_Apps_Used,Avg_Social,Avg_Productivity,Avg_Gaming
0,Female,483,7.630,7.760,6.390,16.590,2.520,2.470,2.490
1,Male,517,7.760,8.050,6.420,16.700,2.400,2.520,2.460


,Group,Metric,Min,Max,Range,Highest_Group,Lowest_Group
14,Location,Avg_Apps_Used,15.930,17.400,1.470,Los Angeles,Houston
2,Age_Group,Avg_Apps_Used,16.170,17.300,1.130,55–59,35–44
12,Location,Avg_Screen_Time,7.400,8.140,0.740,Chicago,New York
13,Location,Avg_Total_App_Usage,5.980,6.610,0.630,New York,Chicago
0,Age_Group,Avg_Screen_Time,7.570,8.050,0.480,18–24,25–34
5,Age_Group,Avg_Gaming,2.310,2.660,0.350,55–59,45–54
3,Age_Group,Avg_Social,2.250,2.590,0.340,55–59,35–44
4,Age_Group,Avg_Productivity,2.360,2.690,0.330,35–44,55–59
1,Age_Group,Avg_Total_App_Usage,6.250,6.560,0.310,18–24,25–34
16,Location,Avg_Productivity,2.420,2.580,0.160,Los Angeles,Chicago


In [42]:
# ============================================================
# 5. Behavioral relationships: correlation, overplotting, and assumption checks
# ============================================================

relationship_cols = [
    "Age",
    "Daily_Screen_Time_Hours",
    "Total_App_Usage_Hours",
    "Number_of_Apps_Used",
    "Social_Media_Usage_Hours",
    "Productivity_App_Usage_Hours",
    "Gaming_App_Usage_Hours",
    "Entertainment_Hours",
    "App_Fragmentation_Score",
]
relationship_cols = [c for c in relationship_cols if c in df.columns]

corr = df[relationship_cols].corr().round(2)
display(corr)

fig = px.imshow(
    corr,
    text_auto=True,
    aspect="auto",
    title="Correlation matrix: relationship structure among usage variables",
    labels={"color": "Correlation"},
    zmin=-1,
    zmax=1,
)
fig.update_layout(height=620)
fig.show()

screen_corr = (
    corr["Daily_Screen_Time_Hours"]
    .drop("Daily_Screen_Time_Hours")
    .sort_values(key=lambda s: s.abs(), ascending=False)
    .reset_index()
)
screen_corr.columns = ["Variable", "Correlation_with_Screen_Time"]
display(screen_corr)

# Overplotting-aware scatter: alpha + marginal distributions
fig = px.scatter(
    df,
    x="Number_of_Apps_Used",
    y="Daily_Screen_Time_Hours",
    color="Dominant_Lifestyle",
    opacity=0.45,
    marginal_x="histogram",
    marginal_y="histogram",
    hover_data=["Age", "Gender", "Location", "Total_App_Usage_Hours"],
    title="Apps used vs. daily screen time: app count alone does not explain usage intensity",
    labels={
        "Number_of_Apps_Used": "Number of apps used",
        "Daily_Screen_Time_Hours": "Daily screen time (hours)",
    },
)
fig.update_layout(height=620)
fig.show()

# Binned average to reveal the trend without being distracted by individual points
bins = pd.cut(df["Number_of_Apps_Used"], bins=[0, 10, 20, 30, 40, 50, 60], include_lowest=True)
binned_apps = (
    df.assign(App_Count_Bin=bins)
    .groupby("App_Count_Bin", observed=True)
    .agg(
        Users=("User_ID", "count"),
        Avg_Screen_Time=("Daily_Screen_Time_Hours", "mean"),
        Median_Screen_Time=("Daily_Screen_Time_Hours", "median"),
        Avg_Total_App_Usage=("Total_App_Usage_Hours", "mean"),
    )
    .reset_index()
)
binned_apps["App_Count_Bin"] = binned_apps["App_Count_Bin"].astype(str)
display(binned_apps.round(2))

fig = px.line(
    binned_apps,
    x="App_Count_Bin",
    y=["Avg_Screen_Time", "Median_Screen_Time"],
    markers=True,
    title="Binned app count view: compare central tendency across app-count ranges",
    labels={"value": "Daily screen time (hours)", "App_Count_Bin": "Number of apps used bin", "variable": "Statistic"},
)
fig.update_layout(height=430)
fig.show()

,Age,Daily_Screen_Time_Hours,Total_App_Usage_Hours,Number_of_Apps_Used,Social_Media_Usage_Hours,Productivity_App_Usage_Hours,Gaming_App_Usage_Hours,Entertainment_Hours,App_Fragmentation_Score
Age,1.000,-0.020,-0.000,-0.000,-0.010,0.010,-0.010,-0.020,-0.010
Daily_Screen_Time_Hours,-0.020,1.000,0.000,0.020,0.030,0.030,-0.010,0.010,-0.030
Total_App_Usage_Hours,-0.000,0.000,1.000,0.040,0.020,-0.010,-0.070,-0.030,0.480
Number_of_Apps_Used,-0.000,0.020,0.040,1.000,0.020,-0.010,0.020,0.030,-0.640
Social_Media_Usage_Hours,-0.010,0.030,0.020,0.020,1.000,-0.080,0.010,0.710,0.010
Productivity_App_Usage_Hours,0.010,0.030,-0.010,-0.010,-0.080,1.000,0.030,-0.030,0.010
Gaming_App_Usage_Hours,-0.010,-0.010,-0.070,0.020,0.010,0.030,1.000,0.710,-0.030
Entertainment_Hours,-0.020,0.010,-0.030,0.030,0.710,-0.030,0.710,1.000,-0.020
App_Fragmentation_Score,-0.010,-0.030,0.480,-0.640,0.010,0.010,-0.030,-0.020,1.000


,Variable,Correlation_with_Screen_Time
0,Social_Media_Usage_Hours,0.030
1,Productivity_App_Usage_Hours,0.030
2,App_Fragmentation_Score,-0.030
3,Age,-0.020
4,Number_of_Apps_Used,0.020
5,Gaming_App_Usage_Hours,-0.010
6,Entertainment_Hours,0.010
7,Total_App_Usage_Hours,0.000


,App_Count_Bin,Users,Avg_Screen_Time,Median_Screen_Time,Avg_Total_App_Usage
0,"(-0.001, 10.0]",272,7.400,7.350,6.280
1,"(10.0, 20.0]",376,8.010,8.110,6.350
2,"(20.0, 30.0]",352,7.600,7.810,6.570


In [43]:
# ============================================================
# 6. EDA insight inventory and dashboard requirements extraction
# ============================================================

# Create reusable insight candidates without touching clustering/PCA from notebook 04.
location_top = location_summary.iloc[0]
location_bottom = location_summary.iloc[-1]
lifestyle_top = lifestyle_summary.sort_values("Users", ascending=False).iloc[0]
strongest_corr = screen_corr.iloc[0]
strongest_group_diff = comparison_scan.iloc[0]

insights = pd.DataFrame([
    {
        "Insight_ID": "EDA-01",
        "Insight": "Heavy screen time is a major behavioral segment",
        "Evidence": f"{heavy_extreme_rate:.1f}% of users belong to Heavy or Extreme screen-time groups.",
        "Dashboard_Use": "Hero KPI + highlighted distribution",
        "Recommended_Visual": "Segmented horizontal bar / annotated distribution",
        "DataViz_Concept": "Preattentive highlight, redundant encoding, position/length",
    },
    {
        "Insight_ID": "EDA-02",
        "Insight": "Average alone hides segment concentration",
        "Evidence": f"Mean screen time is {avg_screen:.2f}h, but the segment view shows a large high-usage tail.",
        "Dashboard_Use": "Opening explanation for why distribution is needed",
        "Recommended_Visual": "Histogram or density with threshold annotations",
        "DataViz_Concept": "Distribution visualization, annotation, threshold reference lines",
    },
    {
        "Insight_ID": "EDA-03",
        "Insight": "Digital lifestyles are relatively balanced",
        "Evidence": f"Largest dominant lifestyle: {lifestyle_top['Dominant_Lifestyle']} with {lifestyle_top['Share_%']:.1f}% of users.",
        "Dashboard_Use": "Lifestyle overview page",
        "Recommended_Visual": "Sorted horizontal bar rather than donut",
        "DataViz_Concept": "Length on common scale, categorical comparison",
    },
    {
        "Insight_ID": "EDA-04",
        "Insight": "Category usage columns should be treated as behavioral intensities",
        "Evidence": "Social + Productivity + Gaming is not always equal to Total_App_Usage_Hours.",
        "Dashboard_Use": "Methodology note to avoid false part-to-whole interpretation",
        "Recommended_Visual": "Category intensity comparison, not total composition pie",
        "DataViz_Concept": "Expressiveness: do not encode relationships not guaranteed by the data",
    },
    {
        "Insight_ID": "EDA-05",
        "Insight": "Location differences are visible in screen time",
        "Evidence": f"{location_top['Location']} has the highest average screen time ({location_top['Avg_Screen_Time']:.2f}h), while {location_bottom['Location']} is lowest ({location_bottom['Avg_Screen_Time']:.2f}h).",
        "Dashboard_Use": "Location comparison card / filter",
        "Recommended_Visual": "Sorted horizontal bar or map-linked ranking",
        "DataViz_Concept": "Position/length, direct labeling, comparison",
    },
    {
        "Insight_ID": "EDA-06",
        "Insight": "App count has limited explanatory power by itself",
        "Evidence": f"Correlation of {strongest_corr['Variable']} with screen time is {strongest_corr['Correlation_with_Screen_Time']:.2f}; app count should be interpreted with other variables.",
        "Dashboard_Use": "Behavioral relationship page",
        "Recommended_Visual": "Scatter with opacity + binned average line",
        "DataViz_Concept": "Overplotting handling, relationship visualization, focus + context",
    },
    {
        "Insight_ID": "EDA-07",
        "Insight": "Some group differences are stronger than others",
        "Evidence": f"Largest scanned group range: {strongest_group_diff['Metric']} across {strongest_group_diff['Group']} groups, range = {strongest_group_diff['Range']:.2f}.",
        "Dashboard_Use": "Prioritize which demographic filters deserve visual space",
        "Recommended_Visual": "Heatmap / small multiples for strong group effects",
        "DataViz_Concept": "Faceting, small multiples, visual hierarchy",
    },
])

display(insights)

# Export insight inventory for docs and later notebook 05
insights_csv = INSIGHT_DIR / "eda_insight_inventory.csv"
insights_md = INSIGHT_DIR / "eda_insight_inventory.md"
insights.to_csv(insights_csv, index=False)
def _df_to_markdown(df):
    cols = list(df.columns)
    header = "| " + " | ".join(cols) + " |"
    separator = "| " + " | ".join(["---"] * len(cols)) + " |"
    rows = [
        "| " + " | ".join(str(v) for v in row) + " |"
        for row in df.itertuples(index=False, name=None)
    ]
    return "\n".join([header, separator, *rows])

insights_md.write_text(_df_to_markdown(insights), encoding="utf-8")

# Export useful summary tables for dashboard design and Power BI/Streamlit cross-checking
segment_summary.to_csv(INSIGHT_DIR / "eda_screen_time_segments.csv", index=False)
lifestyle_summary.to_csv(INSIGHT_DIR / "eda_lifestyle_summary.csv", index=False)
age_summary.to_csv(INSIGHT_DIR / "eda_age_summary.csv", index=False)
location_summary.to_csv(INSIGHT_DIR / "eda_location_summary.csv", index=False)
gender_summary.to_csv(INSIGHT_DIR / "eda_gender_summary.csv", index=False)
comparison_scan.to_csv(INSIGHT_DIR / "eda_group_difference_scan.csv", index=False)
screen_corr.to_csv(INSIGHT_DIR / "eda_screen_time_correlations.csv", index=False)

print("Exported EDA insight inventory and summary tables to:")
print(INSIGHT_DIR)

print("\nRecommended next step:")
print("Use 05_visual_prototyping_storytelling.ipynb to turn these insight candidates into final dashboard visuals and narrative flow.")

,Insight_ID,Insight,Evidence,Dashboard_Use,Recommended_Visual,DataViz_Concept
0,EDA-01,Heavy screen time is a major behavioral segment,48.8% of users belong to Heavy or Extreme scre...,Hero KPI + highlighted distribution,Segmented horizontal bar / annotated distribution,"Preattentive highlight, redundant encoding, po..."
1,EDA-02,Average alone hides segment concentration,"Mean screen time is 7.70h, but the segment vie...",Opening explanation for why distribution is ne...,Histogram or density with threshold annotations,"Distribution visualization, annotation, thresh..."
2,EDA-03,Digital lifestyles are relatively balanced,Largest dominant lifestyle: Productivity Focus...,Lifestyle overview page,Sorted horizontal bar rather than donut,"Length on common scale, categorical comparison"
3,EDA-04,Category usage columns should be treated as be...,Social + Productivity + Gaming is not always e...,Methodology note to avoid false part-to-whole ...,"Category intensity comparison, not total compo...",Expressiveness: do not encode relationships no...
4,EDA-05,Location differences are visible in screen time,Chicago has the highest average screen time (8...,Location comparison card / filter,Sorted horizontal bar or map-linked ranking,"Position/length, direct labeling, comparison"
5,EDA-06,App count has limited explanatory power by itself,Correlation of Social_Media_Usage_Hours with s...,Behavioral relationship page,Scatter with opacity + binned average line,"Overplotting handling, relationship visualizat..."
6,EDA-07,Some group differences are stronger than others,Largest scanned group range: Avg_Apps_Used acr...,Prioritize which demographic filters deserve v...,Heatmap / small multiples for strong group eff...,"Faceting, small multiples, visual hierarchy"


Exported EDA insight inventory and summary tables to:
/Users/hoangquannguyen/Documents/Project/Dataviz/Group14_IT4023E_Dataviz_project/reports/insights

Recommended next step:
Use 05_visual_prototyping_storytelling.ipynb to turn these insight candidates into final dashboard visuals and narrative flow.


## 7. What this notebook should feed into later steps

This EDA notebook produces concrete design inputs for the rest of the project:

- **For `04_advanced_analysis_clustering_pca.ipynb`:** use the relationship patterns as motivation for clustering and PCA, but keep cluster modeling outside this notebook.
- **For `05_visual_prototyping_storytelling.ipynb`:** use `reports/insights/eda_insight_inventory.csv` to choose final chart prototypes and narrative order.
- **For Streamlit:** turn EDA questions into filters, linked views, and drill-down components.
- **For Power BI:** use exported summary tables to validate KPI cards, ranking charts, and demographic comparisons.
